# 第 11 天 · 多工具与并行调用

> 📓 **这是第 11 天 的 Notebook 版**，内容和[课程网页](https://pollux1314.github.io/agent-course/#/day/11)一样。
> 先运行第一格「运行环境准备」，再从上往下逐格 Shift+Enter。你改的代码自动存在这个浏览器里；想恢复原样，在左侧文件列表右键这个文件 → Delete，刷新后会重新出现原版。
>
> 浏览器里的 Python（Pyodide）的限制：`input()`、`subprocess`、本地文件和数据库要用本地 `uv run`；访问外部网站需要对方允许跨域（DeepSeek、通义、GitHub API、httpbin 都可以）。本地脚本里的 `asyncio.run(main())` 在这里已改成 `await main()`；输出里看到 `SystemExit: ...` 是脚本主动退出（比如没填 Key），不是环境坏了。

In [ ]:
# ⚙️ 运行环境准备：打开 Notebook 后先运行这一格（Shift+Enter）
%pip install -q openai pydantic python-dotenv
import sys
sys.argv = ["main.py"]  # 课程脚本里读命令行参数的地方走默认值
try:  # 同步 httpx / openai 在浏览器里走同步 XHR（和课程网页的运行器一致）
    import httpx._transports.jsfetch as _jf
    _jf.can_run_sync = lambda: False
except Exception:
    pass
print("✓ 环境就绪，可以往下运行了")

In [ ]:
# 🔑 这一课要调用大模型：把 Key 填在引号里再运行这一格（只存在这次打开的浏览器内存里，别分享这个 Notebook）
import os
LLM_API_KEY = ""
os.environ["LLM_BASE_URL"] = "https://api.deepseek.com"   # 通义：https://dashscope.aliyuncs.com/compatible-mode/v1
os.environ["LLM_MODEL"] = "deepseek-chat"                 # 通义：qwen-plus
if LLM_API_KEY:
    os.environ["LLM_API_KEY"] = LLM_API_KEY
print("Key：", "已设置" if os.getenv("LLM_API_KEY") else "未设置（会走缺 Key 的报错分支）")

In [ ]:
"""第 11 天 · 多工具与并行调用：天气 + 计算器 + 当前时间
没有 Key 也能跑（假模型照真实协议返回 tool_calls）；填了 LLM_API_KEY 就调用真模型。
要点：① 注册多个工具让模型自己选 ② 一次返回多个 tool_calls 时并发执行 ③ 循环直到模型不再要工具"""
import ast
import asyncio
import json
import operator
import os
import time
from datetime import datetime, timedelta, timezone

from dotenv import load_dotenv
from pydantic import BaseModel, Field, ValidationError

load_dotenv()


# ---------- 1. 三个工具的实现（类比三个 Service 方法） ----------
FAKE_WEATHER = {
    "北京": {"condition": "晴", "temp_c": 12},
    "上海": {"condition": "小雨", "temp_c": 18},
    "深圳": {"condition": "多云", "temp_c": 26},
}


class WeatherArgs(BaseModel):
    city: str = Field(description="城市中文名，如 北京、上海")


async def get_weather(city: str) -> dict:
    await asyncio.sleep(0.5)  # 假装是一次耗时 0.5 秒的网络请求
    data = FAKE_WEATHER.get(city)
    if data is None:
        return {"error": f"没有 {city} 的天气数据，支持：{', '.join(FAKE_WEATHER)}"}
    return {"city": city, **data}


class CalcArgs(BaseModel):
    expression: str = Field(description="只含数字和 + - * / ** ( ) 的算术表达式，如 (18-12)*3")


_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}


def _eval(node):
    """只允许数字和四则运算的安全求值；千万别用 eval()，模型给的字符串不可信"""
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval(node.left), _eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval(node.operand))
    raise ValueError("表达式里有不允许的内容")


async def calculator(expression: str) -> dict:
    try:
        return {"expression": expression, "result": _eval(ast.parse(expression, mode="eval").body)}
    except (ValueError, SyntaxError, ZeroDivisionError) as e:
        return {"error": f"无法计算 {expression!r}：{e}"}


class TimeArgs(BaseModel):
    utc_offset: int = Field(default=8, ge=-12, le=14, description="时区相对 UTC 的小时数，北京时间是 8")


async def get_current_time(utc_offset: int = 8) -> dict:
    now = datetime.now(timezone(timedelta(hours=utc_offset)))
    return {"now": now.strftime("%Y-%m-%d %H:%M:%S"), "weekday": now.strftime("%A"),
            "utc_offset": utc_offset}


# ---------- 2. 注册表：一处登记，自动生成 tools 说明书（类比 Spring 扫描 @Component） ----------
REGISTRY = {
    "get_weather": (WeatherArgs, get_weather, "查询城市当前天气和气温。问天气、冷热、带不带伞时用。"),
    "calculator": (CalcArgs, calculator, "精确计算算术表达式。任何需要算数的地方都用它，不要心算。"),
    "get_current_time": (TimeArgs, get_current_time, "获取当前日期、时间和星期。问现在几点、今天几号时用。"),
}

TOOLS = [{"type": "function",
          "function": {"name": name, "description": desc, "parameters": model.model_json_schema()}}
         for name, (model, _, desc) in REGISTRY.items()]


async def run_tool(name: str, arguments: str) -> str:
    if name not in REGISTRY:
        return json.dumps({"error": f"未知工具 {name}"}, ensure_ascii=False)
    args_model, func, _ = REGISTRY[name]
    try:
        args = args_model.model_validate_json(arguments or "{}")
    except ValidationError as e:
        return json.dumps({"error": f"参数不合法：{e.errors()[0]['msg']}"}, ensure_ascii=False)
    return json.dumps(await func(**args.model_dump()), ensure_ascii=False)


# ---------- 3. 模型 ----------
def _call(cid: str, name: str, args: dict) -> dict:
    return {"id": cid, "type": "function",
            "function": {"name": name, "arguments": json.dumps(args, ensure_ascii=False)}}


async def fake_llm(messages: list[dict], tools: list[dict]) -> dict:
    """模拟真实模型的典型行为：第 1 轮并行要 3 个工具；第 2 轮根据结果要计算器；第 3 轮作答"""
    await asyncio.sleep(0.05)
    results = {m["tool_call_id"]: json.loads(m["content"]) for m in messages if m["role"] == "tool"}
    if not results:
        return {"role": "assistant", "content": None, "tool_calls": [
            _call("call_1", "get_weather", {"city": "北京"}),
            _call("call_2", "get_weather", {"city": "上海"}),
            _call("call_3", "get_current_time", {"utc_offset": 8}),
        ]}
    if "call_4" not in results:
        bj, sh = results["call_1"]["temp_c"], results["call_2"]["temp_c"]
        return {"role": "assistant", "content": None,
                "tool_calls": [_call("call_4", "calculator", {"expression": f"({sh}-{bj})*3"})]}
    t, calc = results["call_3"], results["call_4"]
    return {"role": "assistant", "content":
            f"现在是 {t['now']}。上海 {results['call_2']['temp_c']}°C，北京 {results['call_1']['temp_c']}°C，"
            f"上海更暖和；温差乘以 3 等于 {calc['result']}。"}


async def real_llm(messages: list[dict], tools: list[dict]) -> dict:
    from openai import AsyncOpenAI
    client = AsyncOpenAI(api_key=os.environ["LLM_API_KEY"],
                         base_url=os.getenv("LLM_BASE_URL", "https://api.deepseek.com"))
    resp = await client.chat.completions.create(
        model=os.getenv("LLM_MODEL", "deepseek-chat"), messages=messages, tools=tools, temperature=0)
    msg = resp.choices[0].message
    out = {"role": "assistant", "content": msg.content}
    if msg.tool_calls:
        out["tool_calls"] = [{"id": c.id, "type": "function",
                              "function": {"name": c.function.name, "arguments": c.function.arguments}}
                             for c in msg.tool_calls]
    return out


# ---------- 4. 循环：模型要工具就执行（同一轮的多个调用并发），直到它给出最终回答 ----------
MAX_ROUNDS = 5  # 防死循环，第 12 天细讲


async def ask(question: str, llm) -> str:
    messages = [
        {"role": "system", "content": "你是助手。天气、时间、计算都必须调用工具，不要编造或心算。"
                                      "互不依赖的工具调用请在同一轮一起发出。"},
        {"role": "user", "content": question},
    ]
    print(f"用户：{question}")
    for round_no in range(1, MAX_ROUNDS + 1):
        reply = await llm(messages, TOOLS)
        if not reply.get("tool_calls"):
            return reply["content"]
        messages.append(reply)  # 带 tool_calls 的 assistant 消息必须放回去
        calls = reply["tool_calls"]
        print(f"\n第 {round_no} 轮：模型一次要了 {len(calls)} 个工具")
        for c in calls:
            print(f"  → {c['id']}: {c['function']['name']}({c['function']['arguments']})")

        start = time.perf_counter()
        # 并发执行（类比 CompletableFuture.allOf），gather 返回顺序和传入顺序一致
        results = await asyncio.gather(*(run_tool(c["function"]["name"], c["function"]["arguments"])
                                         for c in calls))
        print(f"  ⏱ 并发执行耗时 {time.perf_counter() - start:.2f}s")
        for c, result in zip(calls, results):
            print(f"  ← {c['id']}: {result}")
            # 每个 tool_call 都必须有一条对应 tool_call_id 的回复，少一条 API 就报 400
            messages.append({"role": "tool", "tool_call_id": c["id"], "content": result})
    return f"超过 {MAX_ROUNDS} 轮还没得出答案，停止。"


async def main():
    llm = real_llm if os.getenv("LLM_API_KEY") else fake_llm
    print("使用", "真模型" if llm is real_llm else "假模型（没填 Key）", "\n")
    answer = await ask("北京和上海现在哪个更暖和？顺便告诉我现在几点，再算一下两地温差乘以 3。", llm)
    print("\n最终回答：", answer)

    print("\n--- 计算器的安全性测试 ---")
    print(await run_tool("calculator", '{"expression": "2**10 + 1/4"}'))
    print(await run_tool("calculator", '{"expression": "__import__(\'os\').system(\'ls\')"}'))
    print(await run_tool("calculator", '{"expression": "1/0"}'))


if __name__ == "__main__":
    await main()  # 本地脚本写 asyncio.run(...)，Notebook 里直接 await